In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# Add current dir to path to import sparse_classifier
sys.path.append(os.path.abspath("."))
from sparse_classifier import SparseClassifier
from conquer3d.data.dataset.digit3d import Digit3D
from conquer3d.data.collate.mesh import bmesh_collate_fn
from conquer3d.conversion.mesh import mesh2sparse
from torchsparse import SparseTensor
import conquer3d as c3d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print("Loading Dataset...")
test_dataset = Digit3D(root="~/.conquer3d/", train=False, download=False, cached=True)

print("Loading Model...")
model = SparseClassifier(num_classes=10).to(device)
ckpt_path = "sparse_classification.pt"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Loaded weights from {ckpt_path}")
else:
    print(f"Warning: {ckpt_path} not found. Using untrained weights.")
model.eval()

In [ ]:
fig = plt.figure(figsize=(20, 10), dpi=300)

for i in range(10):
    # 1. Get sample
    batch = bmesh_collate_fn([test_dataset[i]])
    bmesh, label = batch
    
    bmesh = bmesh.cuda(non_blocking=True)
    bmesh.vertices = bmesh.vertices.float()
    
    batched_coords, batched_sdf = mesh2sparse(bmesh, res=[32, 32, 32], grid_bound=1.2, iso=0.0)
    batched_coords = batched_coords.to(device)
    batched_sdf = batched_sdf.to(device)
    
    # 2. Run Inference
    x = SparseTensor(coords=batched_coords.contiguous(), feats=batched_sdf.contiguous())
    with torch.no_grad():
        logits = model(x)
        pred_class = torch.argmax(logits, dim=1).item()
        
    # 3. Reconstruct Mesh using Conquer3D
    try:
        unique_vertices, local_voxels, merged_sdfs = c3d.data_structure.sparse2mesh_topology(
            batched_coords, batched_sdf, grid_min=[-1.2, -1.2, -1.2], grid_max=[1.2, 1.2, 1.2], res=[32, 32, 32]
        )
    except AttributeError:
        # Fallback to the new api path if it was refactored
        unique_vertices, local_voxels, merged_sdfs = c3d.conversion.grid.sparse2voxel(
            batched_coords, batched_sdf, grid_min=[-1.2, -1.2, -1.2], grid_max=[1.2, 1.2, 1.2], res=[32, 32, 32]
        )
        
    vert, tri, _, _ = c3d.ops.diff_marching_cubes(unique_vertices, local_voxels, merged_sdfs, iso=0.0)
    
    vert_np = vert.detach().cpu().numpy()
    tri_np = tri.detach().cpu().numpy()
    
    # 4. Visualize
    ax = fig.add_subplot(2, 5, i + 1, projection='3d')
    ax.plot_trisurf(vert_np[:, 0], vert_np[:, 1], vert_np[:, 2], triangles=tri_np, cmap='viridis', edgecolor='none')
    
    expected_class = label.item() if isinstance(label, torch.Tensor) else label
    ax.set_title(f"Predicted: {pred_class} | Expected: {expected_class}")
    ax.axis('off')

plt.tight_layout()
plt.show()